# Z3-Rust 17 — Théorie des tableaux : `Select`, `Store` et axiomes de McCarthy

**Navigation** : [Index Z3-Python](README.md) | [Index SMT](../README.md) | [Index SymbolicAI](../../README.md) | [Série Z3 C#](../Z3-Linq2Z3/README.md) | [<< 16e - Optimize](Z3-16e-Meal-Planner-Optimize-Python.ipynb) | [→ 18 - Sudoku modes](Z3-18-Sudoku-Modes-Python.ipynb) | [Jumeau C#](Z3-17-Array-Theory-CSharp.ipynb) | [Version Python](Z3-17-Array-Theory-Python.ipynb)

> **Solveur** : Z3 — crate Rust `z3` 0.23.0, exécuté dans le noyau `rust` (evcxr). Le portage suit le **même fil pédagogique** que le jumeau Python : même modèle, mêmes contraintes, mêmes verdicts.

## Objectifs d'apprentissage

À la fin de ce notebook, vous saurez :
1. Modéliser un **tableau symbolique** avec la théorie des tableaux de Z3 (`Array` sort, `Select`, `Store`).
2. Énoncer et **vérifier les axiomes de McCarthy** (_read-over-write_) qui fondent la sémantique du `Store`.
3. Résoudre des contraintes sur les **contenus** d'un tableau (tableau trié, tableau égal à un autre, permutation) sans énumérer les valeurs.

Ce notebook est le port Rust (`crate z3`) du carnet Python [`Z3-17-Array-Theory-Python.ipynb`](Z3-17-Array-Theory-Python.ipynb), dont le jumeau C# est [`Z3-17-Array-Theory-CSharp.ipynb`](Z3-17-Array-Theory-CSharp.ipynb) — il introduit une **théorie SMT distincte** des notebooks précédents (Booléens, arithmétique, chaînes) : la théorie des tableaux, où un tableau est vu comme une **fonction** Index → Élément.

> **Différence de moteur, à connaître** : le jumeau Python utilise le paquet PyPI `z3-solver` (pyz3 4.16.0), le jumeau C# le paquet NuGet `Microsoft.Z3` (4.12.2), et ce jumeau la crate Rust `z3` 0.23.0, qui lie la bibliothèque native **Z3 5.3.3** du poste. Les trois côtés branchent le **même cœur Z3** par trois bindings officiels ; le cœur est donc différent d'une version à l'autre, la sémantique identique — c'est ce que mesure le registre de conformité différentielle (`scripts/notebook_tools/rust_conformance.d/z3-17-array-theory.yaml`).

In [2]:
:dep z3 = "0.23.0"

// Imports et vérification de l'environnement.
//
// Le noyau `rust` (evcxr) execute chaque cellule dans le meme etat persistant :
// les `fn` persistent d'une cellule a l'autre, les variables locales (`let`) et
// les `use` non. D'ou le patron du carnet : chaque cellule de calcul definit
// ses propres fonctions (qui persistent) et reconstruit son etat local a
// partir des items persistants.

use z3::ast::{Array, Ast, Int};
use z3::{SatResult, Solver};

println!("Imports OK : z3 crate 0.23.0 (libz3 {})", z3::full_version());

Imports OK : z3 crate 0.23.0 (libz3 5.3.3.0 430d19e697e2ec9bad79ff210d41b4a7a5fcfa3e 430d19e)


## 1. Qu'est-ce que la théorie des tableaux ?

Un **tableau** en SMT n'est pas une structure de données mutable : c'est une **fonction totale** `Index → Élément`. Z3 expose cette théorie via le sort `Array(I, E)` et deux opérateurs canoniques :

| Opérateur | Signature | Sens |
|-----------|-----------|------|
| `Select(A, i)` | `Array(I,E) × I → E` | lire l'élément d'indice `i` (`A[i]`) |
| `Store(A, i, v)` | `Array(I,E) × I × E → Array(I,E)` | renvoyer un **nouveau** tableau égal à `A` sauf en `i` où il vaut `v` |

En Rust, ces deux opérateurs sont des **méthodes** de la crate `z3` : `a.select(&i)` pour la lecture — elle renvoie un `Dynamic`, qu'il faut ramener au type `Int` par `.as_int().unwrap()` — et `a.store(&i, &v)` pour l'écriture, qui renvoie un **nouveau** `Array` sans toucher à `a`.

Deux conséquences essentielles :

- **Immuable** : `Store` ne modifie pas `A`, il produit un autre tableau. C'est la sémantique *read-over-write* des tableaux fonctionnels.
- **Axiomes de McCarthy** : la théorie est définie par deux axiomes (vérifiés ci-dessous) qui lient `Select` et `Store`. Le solveur les utilise pour raisonner sur les contenus sans énumérer.

On modélise ainsi des structures indexées (tampons, mémoires, grilles) et on pose des contraintes sur leurs **contenus** — par exemple « le tableau est trié », « deux tableaux coïncident partout sauf en un indice » — que le solveur satisfait ou prouve impossibles.

## 2. Premier terrain — `Select` : contraindre les éléments d'un tableau symbolique

Construisons un tableau symbolique `a : Array(Int, Int)` et exigeons qu'il vérifie quatre contraintes ponctuelles : `a[0] = 1`, `a[1] = 2`, `a[2] > a[0]`, et `a[3] = a[1] + a[2]`. Z3 doit trouver un tableau (une fonction) satisfaisant tout cela.

En Rust, le tableau se déclare par `Array::new_const("a", &tri_int, &tri_int)` et chaque lecture par `a.select(&Int::from_i64(k))` :

```rust
s.assert(&Ast::eq(&element(&a, 0), &Int::from_i64(1)));
s.assert(&element(&a, 2).gt(&element(&a, 0)));
```

In [3]:
// Exemple 1 : Select — contraindre les éléments d'un tableau symbolique.
use z3::ast::{Array, Ast, Int};
use z3::{SatResult, Solver};

// Le `select` de la crate renvoie un `Dynamic` ; `as_int()` le ramène au type
// `Int` attendu par les contraintes arithmétiques.
fn element(a: &Array, i: i64) -> Int {
    a.select(&Int::from_i64(i)).as_int().unwrap()
}

let s = Solver::new();

// Un tableau symbolique d'entiers indexé par des entiers.
let tri_int = Int::from_i64(0).get_sort();
let a = Array::new_const("a", &tri_int, &tri_int);

// Contraintes ponctuelles sur les contenus via Select.
s.assert(&Ast::eq(&element(&a, 0), &Int::from_i64(1)));
s.assert(&Ast::eq(&element(&a, 1), &Int::from_i64(2)));
s.assert(&element(&a, 2).gt(&element(&a, 0)));      // a[2] > a[0]
s.assert(&Ast::eq(&element(&a, 3), &Int::add(&[&element(&a, 1), &element(&a, 2)])));  // a[3] = a[1] + a[2]

let statut = match s.check() {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Statut : {}", statut);

let m = s.get_model().unwrap();
for k in 0..4 {
    let v = m.eval(&element(&a, k), true).unwrap().as_i64().unwrap();
    println!("a[{}] = {}", k, v);
};

Statut : sat


a[0] = 1


a[1] = 2


a[2] = 21240


a[3] = 21242


### Lecture — le tableau comme fonction partielle

Z3 répond `sat` et produit un **modèle partiel** : il ne décrit `a` qu'aux indices qui apparaissent dans les contraintes (`0, 1, 2, 3`). Ailleurs, `a` est laissée libre (le modèle l'expose souvent comme une fonction par défaut, typiquement `K(Int, 0)` — le tableau constant nul). C'est exactement la puissance de la théorie : on ne déclare **pas** la taille du tableau, on ne déclare **pas** de valeurs par défaut ; on pose des contraintes sur les contenus, et le solveur synthétise une fonction cohérente.

En Rust, `m.eval(&element(&a, k), true)` ne renvoie que la valeur **demandée** : le modèle n'est jamais « vidé » ni énuméré, il est **interrogé** indice par indice.

## 3. Second terrain — les axiomes de McCarthy (`read-over-write`)

La sémantique du `Store` repose sur deux axiomes, dus à John McCarthy (1962) pour le λ-calcul avec tableaux. Soient `A` un tableau, `i, j` deux indices et `v` une valeur :

1. **Axiome d'égalité d'indice** : `Select(Store(A, i, v), i) == v` — après écriture en `i`, la lecture en `i` renvoie la valeur écrite.
2. **Axiome de différence d'indice** : `i != j ⇒ Select(Store(A, i, v), j) == Select(A, j)` — écrire en `i` ne change aucun autre indice.

Ces deux axiomes sont **valides par construction** dans la théorie Z3 : on les vérifie en montrant que leur négation est `unsat`.

En Rust, l'indexation est **symbolique** (`i` et `j` sont des `Int` nommés, pas des littéraux), l'inégalité s'écrit `Ast::ne(&i2, &j2)`, l'implication par `.implies(&egalite)`, et la négation par `.not()`.

In [4]:
// Exemple 2 : vérifier les axiomes de McCarthy (négation unsat).
use z3::ast::{Array, Ast, Int};
use z3::{SatResult, Solver};

// Index symbolique : `select` accepte un indice `Int` (et pas seulement un
// littéral) — c'est ce qui permet d'exprimer l'axiome 2.
fn element_sym(a: &Array, i: &Int) -> Int {
    a.select(i).as_int().unwrap()
}

let tri_int = Int::from_i64(0).get_sort();

// Axiome 1 : Select(Store(A, i, v), i) == v.
// Nier l'axiome : s'il existe un contre-exemple, la négation est sat.
let s1 = Solver::new();
let a1 = Array::new_const("A1", &tri_int, &tri_int);
let i1 = Int::new_const("i1");
let v1 = Int::new_const("v1");
let axiome1 = Ast::eq(&element_sym(&a1.store(&i1, &v1), &i1), &v1);
s1.assert(&axiome1.not());
let statut1 = match s1.check() {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Axiome 1 (negation doit etre UNSAT) : {}", statut1);

// Axiome 2 : i != j  =>  Select(Store(A, i, v), j) == Select(A, j).
let s2 = Solver::new();
let a2 = Array::new_const("A2", &tri_int, &tri_int);
let i2 = Int::new_const("i2");
let j2 = Int::new_const("j2");
let v2 = Int::new_const("v2");
let indexes_differents = Ast::ne(&i2, &j2);
let egalite = Ast::eq(&element_sym(&a2.store(&i2, &v2), &j2), &element_sym(&a2, &j2));
let axiome2 = indexes_differents.implies(&egalite);
s2.assert(&axiome2.not());
let statut2 = match s2.check() {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Axiome 2 (negation doit etre UNSAT) : {}", statut2);

Axiome 1 (negation doit etre UNSAT) : unsat


Axiome 2 (negation doit etre UNSAT) : unsat


### Lecture — pourquoi ces axiomes sont `unsat` à la négation

Les deux négations sont `unsat` : **aucun** contre-exemple n'existe. Cela confirme que la théorie des tableaux de Z3 **incorpore** les axiomes de McCarthy — le solveur n'a pas besoin qu'on les ajoute explicitement, ils font partie de la définition du sort `Array`. C'est ce qui permet à Z3 de raisonner sur des suites de `Store` (une mémoire qui écrit plusieurs fois) de manière **déductive** plutôt qu'en énumérant les cas.

Les deux axiomes se lisent dans le code : l'axiome 1 est une **égalité** (`Ast::eq`) sur la lecture de l'écriture, l'axiome 2 une **implication** dont l'antécédent est l'inégalité des indices. Les nier, c'est demander au solveur un contre-exemple — et il n'en existe pas.

## 4. Troisième terrain — contraintes globales sur les contenus

Combinons `Select`, `Store` et quantificateurs pour poser une contrainte **globale** : existe-t-il un tableau `t` de taille 5 (`t[0..4]`) strictement **croissant** ? On borne les indices par une contrainte de domaine, puis on exprime la croissance par un quantificateur borné.

In [5]:
// Exemple 3 : synthétiser un tableau croissant de taille 5.
use z3::ast::{Array, Ast, Int};
use z3::{SatResult, Solver};

// La fonction `element` a été définie à la cellule 4 : sous evcxr, les `fn`
// persistent d'une cellule à l'autre — seul l'état local est reconstruit ici.

let s3 = Solver::new();
let tri_int = Int::from_i64(0).get_sort();
let t = Array::new_const("t", &tri_int, &tri_int);

// Croissance stricte sur les indices consécutifs 0..3 (donc t[0] < t[1] < ... < t[4]).
for k in 0..4 {
    s3.assert(&element(&t, k).lt(&element(&t, k + 1)));
}

// On fixe un domaine raisonnable pour obtenir un modèle lisible.
for k in 0..5 {
    s3.assert(&element(&t, k).ge(&Int::from_i64(0)));
    s3.assert(&element(&t, k).le(&Int::from_i64(100)));
}

let statut = match s3.check() {
    SatResult::Sat => "sat",
    SatResult::Unsat => "unsat",
    SatResult::Unknown => "unknown",
};
println!("Statut : {}", statut);

let m = s3.get_model().unwrap();
let mut valeurs = Vec::new();
for k in 0..5 {
    valeurs.push(m.eval(&element(&t, k), true).unwrap().as_i64().unwrap());
}
let rendu: Vec<String> = valeurs.iter().map(|v| v.to_string()).collect();
println!("Tableau croissant synthétisé : [{}]", rendu.join(", "));

// Démonstration Store : construire t2 = t où l'indice 2 est forcé à 42, et vérifier.
// `store` renvoie un NOUVEAU tableau : `t` n'est pas modifié.
let t2 = t.store(&Int::from_i64(2), &Int::from_i64(42));
let v2_2 = m.eval(&element(&t2, 2), true).unwrap().as_i64().unwrap();
let v2_0 = m.eval(&element(&t2, 0), true).unwrap().as_i64().unwrap();
println!("Apres Store(t, 2, 42) -> Select(t2, 2) = {}", v2_2);
println!("Select(t2, 0) inchangé = {} (== Select(t, 0) = {} )", v2_0, valeurs[0]);

Statut : sat


Tableau croissant synthétisé : [92, 93, 94, 95, 96]


Apres Store(t, 2, 42) -> Select(t2, 2) = 42


Select(t2, 0) inchangé = 92 (== Select(t, 0) = 92 )


### Lecture — `Store` produit un nouveau tableau cohérent, lu statut par statut

Le solveur synthétise un tableau croissant — les valeurs exactes sont celles affichées par la cellule précédente : c'est un **témoin arbitraire**, Z3 est libre de choisir la base (le jumeau Python a ainsi obtenu `[92, 93, 94, 95, 96]` sur son exécution ; ce carnet affiche le témoin de la sienne). Ensuite, `t.store(&Int::from_i64(2), &Int::from_i64(42))` crée `t2` : on lit `Select(t2, 2) == 42` (axiome 1) et `Select(t2, 0) == Select(t, 0)` (axiome 2, car `0 != 2`). C'est la sémantique *read-over-write* en action : une suite d'écritures construit une **chaîne de tableaux immuables**, chacun dérivé du précédent, que le solveur sait inspecter sans ambiguïté.

Le modèle se lit aussi **statut par statut** — fixe, dérivé, libre. Au §2 (`a[0]=1`, `a[1]=2` en égalités, `a[2]>a[0]`, `a[3]=a[1]+a[2]`), le modèle rend 1 et 2 **fixés**, `a[2]` **libre** (la seule contrainte est `> 1` : n'importe quelle valeur au-dessus ferait l'affaire, Z3 en choisit une et s'arrête), `a[3]` **dérivé** (l'arithmétique se referme : `a[3] = a[1] + a[2]`). Ici au §4, même structure : la base du tableau croissant est **libre** (Z3 aurait pu choisir une autre base), `t2[2]=42` est **imposé** par le `Store`, `t2[0]` **inchangé** par l'axiome 2. Un modèle sat ne se lit donc pas valeur par valeur mais statut par statut : seule la part fixée et dérivée est une information ; la part libre changera d'une exécution à l'autre sans rien changer au verdict.

## Exercices

Les trois exercices vous font réutiliser `Select`, `Store` et les axiomes de McCarthy sur de nouvelles contraintes. Chaque exercice suit le squelette : déclarer un `Array`, ajouter des contraintes via `select`/`store`, appeler `check()` puis lire le modèle.

**Rappel C.1** : les stubs ne lèvent **jamais** d'erreur — `println!("Exercice a completer")`. Le notebook s'exécute de bout en bout même exercices non résolus.

### Exercice 1 — Deux tableaux égaux partout sauf en un indice

Étant donné deux tableaux `p` et `q` (`Array(Int, Int)`), ajoutez les contraintes pour que `p` et `q` coïncident en tous les indices `0..4` **sauf** en l'indice 2 où ils diffèrent. Vérifiez `sat` et affichez `p[2]` vs `q[2]` (ils doivent être différents) ainsi que `p[0]` vs `q[0]` (ils doivent être égaux).

**Indice** : `Select(p, k) == Select(q, k)` pour `k != 2`, et `Select(p, 2) != Select(q, 2)`.

In [6]:
// EXERCICE 1 : deux tableaux egaux partout sauf en l'indice 2.
// TODO etudiant :
// 1. p = Array::new_const("p", &tri_int, &tri_int) ; q = Array::new_const("q", &tri_int, &tri_int)
// 2. Pour k dans 0..4 sauf 2 : ajouter Ast::eq(&element(&p, k), &element(&q, k))
// 3. Ajouter Ast::eq(&element(&p, 2), &element(&q, 2)).not()
// 4. s.check(), lire p[2], q[2], p[0], q[0]
println!("Exercice 1 a completer : deux tableaux egaux sauf en l'indice 2.");

Exercice 1 a completer : deux tableaux egaux sauf en l'indice 2.


### Exercice 2 — Prouver un invariant de `Store`

Montrez (par `unsat` de la négation) la propriété : si `B = Store(A, i, v)` et `j != i`, alors `Select(B, j) == Select(A, j)` pour **tout** `j` différent de `i`. Construire la négation et vérifier qu'elle est `unsat`.

**Indice** : c'est l'axiome 2 généralisé ; `B` est défini comme `Store(A, i, v)`, et on nie `Implies(j != i, Select(B, j) == Select(A, j))`.

In [7]:
// EXERCICE 2 : invariant de Store (axiome 2 generalise).
// TODO etudiant :
// 1. A = Array::new_const("A", &tri_int, &tri_int) ; i = Int::new_const("i") ; j = Int::new_const("j") ; v = Int::new_const("v")
// 2. B = A.store(&i, &v)
// 3. propriete = Int::distinct(&[&i, &j]).implies(&Ast::eq(&element_sym(&B, &j), &element_sym(&A, &j)))
// 4. s.assert(&propriete.not()) ; s.check()  -> unsat attendu
println!("Exercice 2 a completer : prouver l'invariant Select(Store(A,i,v),j) == Select(A,j) pour j != i.");

Exercice 2 a completer : prouver l'invariant Select(Store(A,i,v),j) == Select(A,j) pour j != i.


### Exercice 3 — Aucun tableau ne satisfait (réponse `unsat`)

Énoncez une contrainte **impossible** sur un tableau `u : Array(Int, Int)` et vérifiez que le solveur répond `unsat`. Par exemple : `u` strictement croissant sur `0..3` **et** `u[0] > u[4]` (contradictoire). Affichez le verdict.

**Question subsidiaire** : pourquoi `unsat` ici alors que l'exemple 3 était `sat` ?

In [8]:
// EXERCICE 3 : contrainte impossible -> reponse unsat.
// TODO etudiant :
// 1. u = Array::new_const("u", &tri_int, &tri_int)
// 2. Croissance : element(&u, k).lt(&element(&u, k + 1)) pour k dans 0..3
// 3. Contradiction : element(&u, 0).gt(&element(&u, 4))
// 4. s.check()  -> unsat attendu
println!("Exercice 3 a completer : tableau croissant ET u[0] > u[4] -> attendu UNSAT.");

Exercice 3 a completer : tableau croissant ET u[0] > u[4] -> attendu UNSAT.


## Conclusion

Ce notebook introduit la **théorie des tableaux** de Z3 — une théorie SMT distincte des arithmétiques et des chaînes, où un tableau est une fonction `Index → Élément` manipulée via `Select` et `Store`.

| Concept | API Z3 (Python) | API Rust (`crate z3`) | Rôle |
|---------|----------------|------------------------|------|
| Tableau symbolique | `Array('a', IntSort(), IntSort())` | `Array::new_const("a", &tri_int, &tri_int)` | fonction immuable Index → Élément |
| Lecture | `Select(a, i)` | `a.select(&i).as_int().unwrap()` | `a[i]` |
| Écriture | `Store(a, i, v)` | `a.store(&i, &v)` | nouveau tableau égal à `a` sauf en `i` |
| Axiome 1 (McCarthy) | `Select(Store(A,i,v),i) == v` | `Ast::eq(&element_sym(&a.store(&i,&v), &i), &v)` | lu = écrit au même indice |
| Axiome 2 (McCarthy) | `i != j ⇒ Select(Store(A,i,v),j) == Select(A,j)` | `Ast::ne(&i, &j).implies(&…)` | écrit en `i` inchangé ailleurs |

**Quand la théorie devient discriminante** : sur des petites instances, `Select`/`Store` équivalent à de l'énumération. La théorie des tableaux prend tout son sens pour raisonner sur des **contenus sous contraintes globales** (invariants de tampon, égalités partielles, preuves d'impossibilité `unsat`) — exactement le terrain où le solveur SMT distingue un moteur de reconnaissance d'un moteur de **raisonnement déductif**.

## Ce que Rust apporte

Comparé au jumeau Python [`Z3-17-Array-Theory-Python.ipynb`](Z3-17-Array-Theory-Python.ipynb), **les sorties de ce carnet ne montrent aucun écart de capacité** : mêmes valeurs, mêmes verdicts `sat`/`unsat`, même tableau synthétisé (`[92, 93, 94, 95, 96]`), mêmes axiomes de McCarthy vérifiés.

Deux écarts de **forme** seulement, sans portée sémantique :

- le côté Rust **intercale des lignes vides** entre deux impressions — les valeurs sont les mêmes, dans le même ordre ;
- l'**identification de la version du solveur** diffère : `Imports OK : z3-solver version 4.16.0` côté Python, `Imports OK : z3 crate 0.23.0 (libz3 5.3.3.0 …)` côté Rust.

La comparaison **cellule de code par cellule de code** et son détail sont dans le registre de conformité différentielle (`scripts/notebook_tools/rust_conformance.d/z3-17-array-theory.yaml`) et dans la discussion de la PR. Cette mesure décrit **ce carnet et cette exécution**, pas Rust en général.